<a href="https://colab.research.google.com/github/ProgramacionFisica-UQ-2026-1/Canica-mecanica/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
SIMULADOR DE CANICA SOBRE SUPERFICIES TOPOGRÁFICAS
Proyecto Final - Ruta 1
- 3 clases + abstracta + herencia
- NumPy, SciPy (solve_ivp, curve_fit), Matplotlib
- Análisis de datos con ruido y ajuste de parámetros
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from abc import ABC, abstractmethod

# ============================================================
# 1. CLASE ABSTRACTA
# ============================================================
class SistemaFisico(ABC):
    @abstractmethod
    def derivada(self, t, estado):
        pass

    @abstractmethod
    def simular(self, t_span, estado_inicial, t_eval=None):
        pass

# ============================================================
# 2. TERRENO (superficie y gradiente)
# ============================================================
class Terreno:
    def __init__(self, funcion_altura):
        self.altura = funcion_altura

    def gradiente(self, x, y, h=1e-5):
        dzdx = (self.altura(x+h, y) - self.altura(x-h, y)) / (2*h)
        dzdy = (self.altura(x, y+h) - self.altura(x, y-h)) / (2*h)
        return dzdx, dzdy

    def energia_potencial(self, x, y, masa, g):
        return masa * g * self.altura(x, y)

# ============================================================
# 3. DEFINICIÓN DE TERRENOS DISPONIBLES
# ============================================================
TERRENOS = {
    "Montaña clásica": lambda x, y: (
        np.sin(0.8*x)*np.cos(0.8*y) +
        0.5*np.sin(1.5*x)*np.cos(1.2*y) +
        0.3*np.exp(-(x**2+y**2)/4)
    ),
    "Plano inclinado + montículos": lambda x, y: (
        -0.6*x +
        0.8*np.exp(-((x-1)**2+(y-1)**2)/1.2) +
        0.6*np.exp(-((x+0.5)**2+(y+1)**2)/1.0) -
        0.5*np.exp(-((x+1)**2+(y-1)**2)/1.5)
    ),
    "Dos picos y valle central": lambda x, y: (
        1.2*np.exp(-((x+1.5)**2+(y+1.5)**2)/1.5) +
        1.2*np.exp(-((x-1.5)**2+(y-1.5)**2)/1.5) -
        0.8*np.exp(-(x**2+y**2)/2.0) +
        0.1*np.sin(1.2*x)*np.cos(1.2*y)
    ),
    "Pozo de potencial atractivo": lambda x, y: (
        -1.5*np.exp(-(x**2+y**2)/1.2) +
        0.2*np.sin(1.5*x)*np.cos(1.5*y)
    ),
    "Silla de montar": lambda x, y: (
        0.8*(x**2 - y**2)*np.exp(-(x**2+y**2)/6) -
        0.5*np.exp(-((x-1)**2+y**2)/2)
    ),
    "Montaña rugosa": lambda x, y: (
        np.sin(0.8*x)*np.cos(0.8*y) +
        0.5*np.sin(1.5*x)*np.cos(1.2*y) +
        0.3*np.sin(2.2*x)*np.cos(2.0*y) +
        0.2*np.sin(3.0*x)*np.cos(2.5*y) +
        0.3*np.exp(-(x**2+y**2)/4)
    )
}

# ============================================================
# 4. PARTÍCULA (con eventos de parada)
# ============================================================
class PararPorVelocidad:
    def __init__(self, v_umbral):
        self.v_umbral = v_umbral
        self.terminal = True
        self.direction = -1

    def __call__(self, t, estado):
        vx, vy = estado[2], estado[3]
        return np.hypot(vx, vy) - self.v_umbral

class LimiteDominio:
    def __init__(self, limite):
        self.limite = limite
        self.terminal = True
        self.direction = 0

    def __call__(self, t, estado):
        x, y = estado[0], estado[1]
        return max(abs(x), abs(y)) - self.limite

class Particula(SistemaFisico):
    def __init__(self, terreno, masa=1.0, rozamiento=0.1, gravedad=9.8,
                 v_umbral=1e-2, dominio_limite=7.0):
        self.terreno = terreno
        self.masa = masa
        self.rozamiento = rozamiento
        self.gravedad = gravedad
        self.v_umbral = v_umbral
        self.dominio_limite = dominio_limite

    def derivada(self, t, estado):
        x, y, vx, vy = estado
        dzdx, dzdy = self.terreno.gradiente(x, y)
        ax = -self.gravedad * dzdx - self.rozamiento * vx
        ay = -self.gravedad * dzdy - self.rozamiento * vy
        return [vx, vy, ax, ay]

    def simular(self, t_span, estado_inicial, t_eval=None):
        evento_vel = PararPorVelocidad(self.v_umbral)
        evento_lim = LimiteDominio(self.dominio_limite)
        sol = solve_ivp(self.derivada, t_span, estado_inicial,
                        t_eval=t_eval, method='RK45',
                        events=[evento_vel, evento_lim],
                        rtol=1e-6, atol=1e-8)
        return sol.t, sol.y

# ============================================================
# 5. FUNCIONES DE VISUALIZACIÓN Y ANÁLISIS
# ============================================================
def graficar_trayectoria(terreno, trayectoria, titulo, limite=7.0):
    xg = np.linspace(-limite, limite, 100)
    yg = np.linspace(-limite, limite, 100)
    X, Y = np.meshgrid(xg, yg)
    Z = terreno.altura(X, Y)

    plt.figure(figsize=(8, 7))
    plt.contourf(X, Y, Z, levels=30, cmap='terrain', alpha=0.85)
    plt.contour(X, Y, Z, levels=10, colors='black', linewidths=0.3, alpha=0.5)
    plt.plot(trayectoria[:, 0], trayectoria[:, 1], 'r-', lw=2, label='Trayectoria')
    plt.scatter(trayectoria[0, 0], trayectoria[0, 1], c='lime', s=100, edgecolors='k', label='Inicio')
    plt.scatter(trayectoria[-1, 0], trayectoria[-1, 1], c='blue', s=100, edgecolors='k', label='Fin')
    plt.xlim(-limite, limite); plt.ylim(-limite, limite)
    plt.xlabel('x (m)'); plt.ylabel('y (m)')
    plt.title(titulo)
    plt.legend()
    plt.colorbar(label='Altura (m)')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

def graficar_energias(tiempos, energias):
    plt.figure(figsize=(10, 4))
    plt.plot(tiempos, energias[:, 0], label='Cinética', color='orangered')
    plt.plot(tiempos, energias[:, 1], label='Potencial', color='steelblue')
    plt.plot(tiempos, energias[:, 2], 'k--', lw=1.5, label='Total')
    plt.xlabel('Tiempo (s)'); plt.ylabel('Energía (J)')
    plt.title('Evolución de energías')
    plt.legend(); plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

def analisis_avanzado(terreno, dominio_limite=7.0):
    print("\n" + "="*60)
    print("ANÁLISIS AVANZADO: Ajuste de parámetros con curve_fit")
    print("="*60)

    roz_real = 0.12
    g_real = 9.8
    masa = 1.0
    estado0 = [1.0, 1.5, 0.0, 0.0]
    t_max = 10.0
    t_med_full = np.linspace(0, t_max, 150)

    # Datos "reales" + ruido
    particula_real = Particula(terreno, masa, roz_real, g_real, dominio_limite=dominio_limite)
    t_real, y_real = particula_real.simular((0, t_max), estado0, t_eval=t_med_full)

    # El error ocurría aquí: si la simulación se detiene antes de t_max,
    # y_real tiene menos puntos que t_med_full. Sincronizamos los tiempos.
    t_med = t_real
    ruido = 0.03
    x_med = y_real[0] + np.random.normal(0, ruido, size=t_med.shape)
    y_med = y_real[1] + np.random.normal(0, ruido, size=t_med.shape)

    def modelo_ajuste(t, roz, g):
        part_temp = Particula(terreno, masa, roz, g, dominio_limite=dominio_limite)
        _, y_temp = part_temp.simular((0, t_max), estado0, t_eval=t)
        return np.concatenate([y_temp[0], y_temp[1]])

    ydata = np.concatenate([x_med, y_med])
    try:
        popt, pcov = curve_fit(modelo_ajuste, t_med, ydata,
                               p0=[0.2, 9.5], maxfev=6000,
                               bounds=([0.0, 2.0], [2.0, 20.0]))
        perr = np.sqrt(np.diag(pcov))
        roz_aj, g_aj = popt

        modelo_aj = Particula(terreno, masa, roz_aj, g_aj, dominio_limite=dominio_limite)
        _, y_aj = modelo_aj.simular((0, t_max), estado0, t_eval=t_med)

        mse = np.mean((x_med - y_aj[0])**2 + (y_med - y_aj[1])**2)
        mae = np.mean(np.abs(x_med - y_aj[0]) + np.abs(y_med - y_aj[1]))

        print(f"\nRozamiento real : {roz_real:.3f}  →  ajustado : {roz_aj:.4f} ± {perr[0]:.4f}")
        print(f"Gravedad real   : {g_real:.1f}    →  ajustada : {g_aj:.4f} ± {perr[1]:.4f}")
        print(f"\nError cuadrático medio (MSE): {mse:.6f}")
        print(f"Error absoluto medio   (MAE): {mae:.6f}")
        print("\n✅ Análisis completado con scipy.optimize.curve_fit")
    except Exception as e:
        print(f"❌ Error en el ajuste: {e}")

# ============================================================
# 6. PROGRAMA PRINCIPAL
# ============================================================
if __name__ == "__main__":
    # ===== CONFIGURACIÓN DEL USUARIO (cambia aquí los parámetros) =====
    TERRENO_ELEGIDO = "Plano inclinado + montículos"  # Cambia el nombre
    MASA = 1.0
    ROZAMIENTO = 0.1
    GRAVEDAD = 9.8
    POSICION_INICIAL = np.array([-2.0, 1.8])
    VELOCIDAD_INICIAL = np.array([-2.0, -4.0])
    T_MAX = 200.0   # tiempo máximo (la simulación se detendrá sola)
    LIMITE_DOMINIO = 7.0
    # ================================================================

    print("🎯 SIMULADOR DE CANICA SOBRE SUPERFICIE TOPOGRÁFICA")
    print(f"Terreno: {TERRENO_ELEGIDO}")
    print(f"Rozamiento = {ROZAMIENTO}, Gravedad = {GRAVEDAD} m/s²")
    print(f"Posición inicial: ({POSICION_INICIAL[0]}, {POSICION_INICIAL[1]})")
    print(f"Velocidad inicial: ({VELOCIDAD_INICIAL[0]}, {VELOCIDAD_INICIAL[1]}) m/s")
    print("-" * 60)

    # Crear terreno
    funcion_altura = TERRENOS[TERRENO_ELEGIDO]
    terreno = Terreno(funcion_altura)

    # Crear partícula y simular
    particula = Particula(terreno, MASA, ROZAMIENTO, GRAVEDAD,
                          v_umbral=0.005, dominio_limite=LIMITE_DOMINIO)
    estado0 = [POSICION_INICIAL[0], POSICION_INICIAL[1],
               VELOCIDAD_INICIAL[0], VELOCIDAD_INICIAL[1]]
    tiempos, y_data = particula.simular((0, T_MAX), estado0, t_eval=None)

    if y_data.size == 0:
        print("❌ La simulación no produjo datos. Revise parámetros.")
    else:
        estados = y_data.T
        trayectoria = estados[:, :2]

        # Calcular energías
        Ec = 0.5 * MASA * (estados[:, 2]**2 + estados[:, 3]**2)
        Ep = np.array([terreno.energia_potencial(x, y, MASA, GRAVEDAD) for x, y in trayectoria])
        energias = np.column_stack([Ec, Ep, Ec + Ep])

        # Mostrar información de parada
        v_final_val = np.hypot(estados[-1, 2], estados[-1, 3])
        print(f"\n⏱️ Simulación terminada en t = {tiempos[-1]:.2f} s")
        print(f"   Velocidad final: {v_final_val:.4f} m/s")
        if np.abs(trayectoria[-1, 0]) >= LIMITE_DOMINIO - 0.1 or np.abs(trayectoria[-1, 1]) >= LIMITE_DOMINIO - 0.1:
            print("⚠️ La canica salió del dominio.")
        elif v_final_val < 0.005:
            print("✅ La canica se detuvo por rozamiento.")
        else:
            print("⚠️ Se alcanzó el tiempo máximo sin detenerse.")

        # Gráficas
        graficar_trayectoria(terreno, trayectoria, f'Trayectoria - {TERRENO_ELEGIDO}', LIMITE_DOMINIO)
        graficar_energias(tiempos, energias)

        # Análisis avanzado
        analisis_avanzado(terreno, LIMITE_DOMINIO)
